# Foundation — the epistemic model and trust boundary

Component 00, which has no I/O and no credentials. Every cell here runs offline. Replaces `scripts/check_gate1.py`.

This is the component the reviewer's questions actually needed: *"how do requests become trusted rule inputs, how are stale or incorrect graph facts handled?"*

## Setup

Redacted readiness report — `config.py` never prints a secret.

In [1]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")          # Neo4j emits schema hints we handle
import config
print(config.report())

readiness
  neo4j      uri=neo4j+s://<redacted>.databases.neo4j.io
             user fp=6b611cde  db fp=6b611cde
             password fp=58ebef03
             configured=True
  qdrant     url=https://<redacted>.sa-east-1-0.aws.cloud.qdrant.io
             api_key fp=f3da5a0f
             configured=True
  embedding  provider=bedrock  model=amazon.titan-embed-text-v2:0  dim=1024
             configured=True
  trust      policy=v1.0  floor=0.75  min_tier=ingested


## A fact cannot exist without provenance

The stamp is a required parameter with no default, so constructing an unstamped fact is a **type error** rather than a runtime check.

In [2]:
from biomedical.foundation import Fact

try:
    Fact(key="consent_status", value="Withdrawn")
except TypeError as e:
    print("TypeError (correct):", e)

TypeError (correct): Fact.__init__() missing 3 required positional arguments: 'stamp', 'validity', and 'recorded_at'


## Confidence cannot be authored

`Confidence` is a property of a registered `ResolutionMethod`, never a float. v1 carried 30 hand-asserted `1.0` values with no derivation.

In [3]:
from biomedical.foundation import Confidence, ResolutionMethod

try:
    Confidence(method=0.95)
except TypeError as e:
    print("TypeError (correct):", e)

print()
for m in ResolutionMethod:
    print(f"  {m.label:<36} {m.default_confidence:.2f}")

TypeError (correct): confidence requires a registered ResolutionMethod

  exact_identifier                     1.00
  normalised_exact                     0.95
  unique_ingredient_after_dose_strip   0.90
  exact_synonym_match                  0.90
  curated_abbreviation_map             0.80
  single_fuzzy_above_threshold         0.60
  multiple_candidates                  0.30
  inverse_of_relation                  1.00
  shared_name_token                    0.90
  human_review                         1.00


## The trust boundary

Seven reasons a fact may be refused as a rule input. Staleness is **per source** — an adverse-event report and a gene symbol assignment do not age at the same rate.

In [4]:
from datetime import datetime, timedelta, timezone
from biomedical.foundation import (Fact, ReviewState, Tier, TrustPolicy, ValidityInterval,
                                   derived, ingested, is_eligible_rule_input)

NOW = datetime.now(timezone.utc)
policy = TrustPolicy()

def fact(**kw):
    return Fact(key="k", value="v",
                stamp=kw.pop("stamp", ingested("openfda:faers", "2026-09", NOW)),
                validity=kw.pop("validity", ValidityInterval(valid_from=NOW - timedelta(days=1))),
                recorded_at=NOW, **kw)

cases = {
    "fresh ingested fact": fact(),
    "derived, low confidence": fact(stamp=derived("resolver", "1.0", NOW,
                                    ResolutionMethod.MULTI_CANDIDATE, "3 candidates")),
    "stale (100d, faers TTL 90d)": fact(validity=ValidityInterval(valid_from=NOW - timedelta(days=100))),
    "awaiting review": fact(review_state=ReviewState.NEEDS_REVIEW),
    "release-pinned source, 5000d old": fact(stamp=ingested("hgnc", "current", NOW),
                                    validity=ValidityInterval(valid_from=NOW - timedelta(days=5000))),
}
for label, f in cases.items():
    v = is_eligible_rule_input(f, policy, NOW)
    print(f"  {label:<36} {v.describe()}")

  fresh ingested fact                  eligible (policy v1.0)
  derived, low confidence              ineligible: tier_too_weak, confidence_below_floor
  stale (100d, faers TTL 90d)          ineligible: stale
  awaiting review                      ineligible: awaiting_review
  release-pinned source, 5000d old     eligible (policy v1.0)


## The fail-open fix

`CANNOT_EVALUATE` separates *"evaluated, found nothing"* from *"could not evaluate"*. Conflating them is the most dangerous behaviour a governance system can have.

In [5]:
from biomedical.foundation import RuleOutcome, assemble

for o in RuleOutcome:
    print(f"  {o.value:<18} is_safe={o.is_safe!s:<6} blocks={o.blocks}")

print()
pending = fact(review_state=ReviewState.NEEDS_REVIEW)
bundle = assemble([pending], policy, NOW)
found, missing = bundle.require("k")
print(bundle.describe())
print("  rule received:", found, " missing:", missing)
print("  -> the rule must return CANNOT_EVALUATE, never NO_VIOLATION")
for w in bundle.withheld:
    print("  withheld:", w.describe())

  violation          is_safe=False  blocks=True
  no_violation       is_safe=True   blocks=False
  cannot_evaluate    is_safe=False  blocks=True

FactBundle(0 eligible, 1 withheld, hash=4f53cda18c2b, policy=v1.0)
  rule received: []  missing: ['k']
  -> the rule must return CANNOT_EVALUATE, never NO_VIOLATION
  withheld: k withheld: awaiting_review


## Evidence parity is structural

The same mechanism that keeps rules honest also proves two evaluation arms received identical evidence. One hash, two problems.

In [6]:
from biomedical.foundation import ParityError, assert_parity

facts = [fact(), ingested and fact()]
a = assemble(facts, policy, NOW)
b = assemble(list(reversed(facts)), policy, NOW)
print("same evidence, different order -> same hash:", a.content_hash == b.content_hash)

strict = assemble(facts, TrustPolicy(min_tier=Tier.ASSERTED), NOW)
try:
    assert_parity(a, strict)
except ParityError as e:
    print("ParityError (correct):", str(e)[:110])

same evidence, different order -> same hash: True
ParityError (correct): evidence parity violated: arms would receive different fact sets (['4f53cda18c2b', 'fe7a98911170']). Refusing 


## Claims discipline

A claim that cannot state its own limits is rejected at build time. The lint pass also fails on bare rates and on "proof" applied to an empirical result — including v1's own headline.

In [7]:
from biomedical.foundation import Claim, ClaimCategory, ClaimError, lint_public_text

try:
    Claim(claim_id="x", claim="A bounded claim", category=ClaimCategory.ACCURACY,
          evidence="manifests/x.json", n=12, does_not_establish="")
except ClaimError as e:
    print("ClaimError (correct):", str(e)[:100])

print()
for text in [
    "The judge scored 58% accuracy.",
    "The judge scored 58% accuracy (n=12, TNR 0.000 [0.00, 0.43]).",
    "Adversarial-proof by architecture, not tuning.",
    "Neurosymbolic AI evaluation framework proving that LLM-as-Judge is worse than no evals.",
]:
    problems = lint_public_text(text)
    verdict = "CLEAN" if not problems else f"{len(problems)} finding(s)"
    print(f"  {verdict:<14} {text[:66]}")
    for p in problems:
        print(f"                   - {p.split(': ', 1)[1]}")

ClaimError (correct): claim 'x': 'does_not_establish' is empty. A claim whose limits cannot be stated must not be publishe

  1 finding(s)   The judge scored 58% accuracy.
                   - rate '58%' published without a visible n or interval
  CLEAN          The judge scored 58% accuracy (n=12, TNR 0.000 [0.00, 0.43]).
  2 finding(s)   Adversarial-proof by architecture, not tuning.
                   - banned phrase 'Adversarial-proof' (claims must be bounded)
                   - banned phrase 'proof' (claims must be bounded)
  1 finding(s)   Neurosymbolic AI evaluation framework proving that LLM-as-Judge is
                   - banned phrase 'proving' (claims must be bounded)


## The claims register

Eight claims, each with what it does **not** establish.

In [8]:
from biomedical.ui.data import claims
print(claims().render())

== ACCURACY ==
[accuracy] An unanchored LLM judge behaved as a constant-FAIL predictor on this set
    evidence: manifests/*_report.json (v1 module E, recomputed)  n=12 TNR 0.000 [0.000, 0.434]
    does not establish: That LLM judges lack predictive value in general. One domain, one model, one prompt style, and only 5 true-negative opportunities.
[accuracy] Giving the judge the same graph facts the rules get raised kappa from 0.273 to 0.833
    evidence: manifests/run_20261011_093151_5b859c_report.json  n=12 kappa 0.833 [0.437, 1.000] vs 0.273 [-0.286, 0.800]
    does not establish: That rules and judges are equivalent. The blind arm's interval includes zero, so even its point estimate is not distinguishable from no skill.
[accuracy] 48% of drug-name resolutions from real sources required human review
    evidence: GRAPH.md, 189 resolutions over 7 condition slices  n=189
    does not establish: A resolution accuracy figure. No gold standard exists for these strings; round-trip agreemen